[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivanderlab/IvanderLab-Colab-Suite/blob/main/Colab_RealtimeVoiceChanger.ipynb)

# 🎙️ W-Okada Realtime Voice Changer (RVC / MMVC) - Colab Free Suite
### ⚡ Powered by `w-okada/voice-changer` & `tg-develop` | Zero-Token Cloudflare Tunnel | Google Colab Free Tier (Tesla T4)

---

### 🎭 Model Suara Pre-Configured (Auto-Installed):
* **Slot 0**: **Mythia Batford** (Indo VTuber - Energetic & Cute) -> Pitch `+12`
* **Slot 1**: **Vestia Zeta** (Hololive ID Gen 3 - Pon Secret Agent) -> Pitch `+12`

---

### 💡 Rekomendasi Settingan WebUI di Colab (Tesla T4):
* **Mode**: `Client` (Streaming audio mikrofon langsung dari browser laptop)
* **f0 Detection**: `rmvpe` (Paling jernih/akurat) atau `fcpe` (Paling cepat/rendah latency)
* **Chunk Size**: `80ms - 128ms` *(Semakin kecil = semakin minim delay)*
* **Extra**: `4096` atau `5s`
* **GPU Selection**: Pilih `cuda:0 (Tesla T4)` pada dropdown GPU di dalam WebUI.
* **Pitch / Tune**:
  * Suara Cowok -> Karakter Cewek/Anime: `+12` (naik 1 oktaf)
  * Suara Cewek -> Karakter Cewek: `0`
  * Suara Cewek -> Karakter Cowok: `-12`

---
### 📋 Panduan 3 Langkah Cepat:
1. Jalankan **CELL 1** (Setup Environment & Binaries Engine)
2. Jalankan **CELL 1.5** (Auto-Install Mythia Batford ke Slot 0 & Vestia Zeta ke Slot 1 via Datacenter Speed ~2-3 detik)
3. Jalankan **CELL 2** (Launch WebUI + Cloudflare Tunnel) -> Buka linknya, klik Slot 0 / Slot 1, langsung ngomong!

In [ ]:
# ==============================================================================
# 📦 CELL 1: SETUP ENVIRONMENT & DOWNLOAD VOICE CHANGER BINARIES
# ==============================================================================
import os
import sys
import time
import json
import codecs
import subprocess
import urllib.request
from google.colab import drive

# 1. Parameter Google Drive (Opsional: Simpan Model Suara RVC Permanen)
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}

print("🚀 [1/2] Menyiapkan environment & mengunduh Voice Changer engine...")

# 2. Install utility sistem & aria2c untuk download kilat
subprocess.run(["apt-get", "-y", "install", "-qq", "aria2", "libportaudio2"], check=False)
subprocess.run(["pip", "install", "-q", "python-dotenv", "pyngrok", "colorama"], check=False)

# 3. Ambil URL rilis terbaru tg-develop/voice-changer
REPO_API = codecs.decode("uggcf://ncv.tvguho.pbz/ercbf/gt-qrirybc/ibvpr-punatre/eryrnfrf/yngrfg", "rot13")
LINUX_PREBUILT = codecs.decode("ibvpr-punatre-yvahk-nzq64-phqn.gne.tm", "rot13")
REAL1 = codecs.decode("ibvpr-punatre", "rot13")
REAL2 = codecs.decode("ZZIPFreireFVB", "rot13")

os.chdir("/content")

# Download Cloudflare Tunnel binary jika belum ada
if not os.path.exists("/content/cloudflared"):
    subprocess.run(["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/content/cloudflared"], check=False)
    subprocess.run(["chmod", "+x", "/content/cloudflared"], check=False)

if not os.path.exists("/content/ML_Program"):
    print("📥 Mengunduh prebuilt CUDA binaries via aria2c (Kecepatan Maksimal)...")
    req = urllib.request.Request(REPO_API, headers={'User-Agent': 'Mozilla/5.0'})
    res = json.loads(urllib.request.urlopen(req).read())

    for asset in res.get('assets', []):
        name = asset.get('name', '')
        if name.startswith(LINUX_PREBUILT):
            url = asset.get('browser_download_url')
            print(f"  ⚡ Downloading {name}...")
            subprocess.run(["aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M", url, "-d", "/content", "-o", name], check=False)

    print("📦 Mengekstrak binaries...")
    subprocess.run(f"cat {LINUX_PREBUILT}.* | tar -xzf -", shell=True, check=True)
    subprocess.run(f"rm -f {LINUX_PREBUILT}.*", shell=True, check=False)

    # Rename folder ke ML_Program
    if os.path.exists(f"/content/{REAL2}"):
        subprocess.run(["mv", f"/content/{REAL2}", "/content/ML_Program"], check=False)
        if os.path.exists(f"/content/ML_Program/{REAL2}"):
            subprocess.run(["mv", f"/content/ML_Program/{REAL2}", "/content/ML_Program/ML_Program"], check=False)

    # 🔧 FIX UBUNTU 24.04 (NOBLE) LIBSTDC++ CONFLICT FOR LIBPORTAUDIO / SOUNDDEVICE
    internal_dir = "/content/ML_Program/_internal"
    if os.path.exists(internal_dir):
        subprocess.run(["rm", "-f"] + [os.path.join(internal_dir, f) for f in os.listdir(internal_dir) if "libstdc++.so" in f], check=False)
        if os.path.exists("/usr/lib/x86_64-linux-gnu/libstdc++.so.6"):
            os.symlink("/usr/lib/x86_64-linux-gnu/libstdc++.so.6", os.path.join(internal_dir, "libstdc++.so.6"))

    print("✅ Binary berhasil diekstrak & dioptimalkan untuk Ubuntu 24.04!")
else:
    print("✅ Engine Voice Changer sudah terpasang!")

# 4. Hubungkan Google Drive jika dipilih
if USE_GOOGLE_DRIVE:
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    drive_model_dir = f"/content/drive/MyDrive/{REAL1}/server/model_dir"
    drive_pretrain_dir = f"/content/drive/MyDrive/{REAL1}/server/pretrain"
    os.makedirs(drive_model_dir, exist_ok=True)
    os.makedirs(drive_pretrain_dir, exist_ok=True)

    # Link folder models ke Google Drive
    if os.path.exists("/content/ML_Program/model_dir") and not os.path.islink("/content/ML_Program/model_dir"):
        subprocess.run(["rm", "-rf", "/content/ML_Program/model_dir"], check=False)
    if os.path.exists("/content/ML_Program/pretrain") and not os.path.islink("/content/ML_Program/pretrain"):
        subprocess.run(["rm", "-rf", "/content/ML_Program/pretrain"], check=False)

    if not os.path.exists("/content/ML_Program/model_dir"):
        os.symlink(drive_model_dir, "/content/ML_Program/model_dir")
    if not os.path.exists("/content/ML_Program/pretrain"):
        os.symlink(drive_pretrain_dir, "/content/ML_Program/pretrain")
    print("💾 Google Drive tersambung! Model suara (.pth & .index) tersimpan aman.")

print("\n🎉 [OK] Setup Selesai! Silakan jalankan CELL 1.5 untuk memasang model RVC instan, lalu jalankan CELL 2.")

🚀 [1/2] Menyiapkan environment & mengunduh Voice Changer engine...
✅ Engine Voice Changer sudah terpasang!
Mounted at /content/drive
💾 Google Drive tersambung! Model suara (.pth & .index) tersimpan aman.

🎉 [OK] Setup Selesai! Silakan jalankan CELL 1.5 untuk memasang model RVC instan, lalu jalankan CELL 2.


In [ ]:
# ==============================================================================
# 📥 CELL 1.5: AUTO-INSTALL RVC MODELS (SLOT 0: MYTHIA BATFORD & SLOT 1: VESTIA ZETA)
# ==============================================================================
import os
import glob
import json
import shutil
import zipfile
import subprocess
import torch

# @markdown ### 🎭 Konfigurasi Model Suara RVC (Auto-Download ke Slot W-Okada)
INSTALL_MYTHIA = True #@param {type:"boolean"}
MYTHIA_URL = "https://huggingface.co/megaaziib/my-rvc-models-collection/resolve/main/MythiaBatford.zip" #@param {type:"string"}
MYTHIA_SLOT = 0 #@param {type:"integer"}
MYTHIA_TUNE = 12 #@param {type:"integer"}

INSTALL_ZETA = True #@param {type:"boolean"}
ZETA_URL = "https://huggingface.co/megaaziib/my-rvc-models-collection/resolve/main/zetaTest.zip" #@param {type:"string"}
ZETA_SLOT = 1 #@param {type:"integer"}
ZETA_TUNE = 12 #@param {type:"integer"}

# @markdown ---
# @markdown **Opsional: Tambah Model Kustom Lain (Slot 2, 3, dst):**
CUSTOM_URL = "" #@param {type:"string"}
CUSTOM_SLOT = 2 #@param {type:"integer"}
CUSTOM_TUNE = 12 #@param {type:"integer"}

def install_rvc_model(url, slot_num, tune, custom_name=None):
    if not url or not url.strip():
        return
    raw_name = os.path.basename(url.split("?")[0])
    base_name = os.path.splitext(raw_name)[0]

    slot_folder = f"/content/ML_Program/model_dir/{slot_num}"
    os.makedirs(slot_folder, exist_ok=True)
    temp_file = f"/content/{raw_name}"

    print(f"\n⚡ [Slot {slot_num}] Mengunduh {raw_name} (Google Datacenter Speed)...")
    subprocess.run(["aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", url, "-d", "/content", "-o", raw_name], check=False)

    pth_file = ""
    index_file = ""

    if raw_name.endswith(".zip") and os.path.exists(temp_file):
        print(f"📦 [Slot {slot_num}] Mengekstrak file zip...")
        with zipfile.ZipFile(temp_file, 'r') as zip_ref:
            for member in zip_ref.namelist():
                fname = os.path.basename(member)
                if not fname:
                    continue
                target_path = os.path.join(slot_folder, fname)
                with zip_ref.open(member) as source, open(target_path, "wb") as target:
                    shutil.copyfileobj(source, target)
                if fname.endswith(".pth"):
                    pth_file = fname
                elif fname.endswith(".index"):
                    index_file = fname
        os.remove(temp_file)
    elif os.path.exists(temp_file):
        dest_path = os.path.join(slot_folder, raw_name)
        shutil.move(temp_file, dest_path)
        if raw_name.endswith(".pth"):
            pth_file = raw_name

    # Scan folder slot jika belum terdeteksi otomatis
    for f in os.listdir(slot_folder):
        if f.endswith(".pth") and not pth_file:
            pth_file = f
        elif f.endswith(".index") and not index_file:
            index_file = f

    # 🧠 DETEKSI CERDAS & PRESISI: ARSITEKTUR MODEL DARI .PTH (RVC v1 vs RVC v2)
    is_v2 = False
    emb_channels = 256
    emb_output_layer = 9
    use_final_proj = True
    sr = 40000
    f0 = True
    full_pth_path = os.path.join(slot_folder, pth_file) if pth_file else ""

    if full_pth_path and os.path.exists(full_pth_path):
        try:
            cpt = torch.load(full_pth_path, map_location="cpu")
            if isinstance(cpt, dict):
                # 1. Cek metadata eksplisit version
                ver = str(cpt.get("version", "")).lower()
                if ver == "v2":
                    is_v2 = True

                # 2. Cek bentuk tensor pembobot asli (Ground Truth)
                if "weight" in cpt:
                    w = cpt["weight"]
                    for k in ["enc_p.emb_phone.weight", "enc_p.text_embedding.weight"]:
                        if k in w and hasattr(w[k], "shape"):
                            if 768 in w[k].shape:
                                is_v2 = True
                            elif 256 in w[k].shape:
                                is_v2 = False
                            break
                    else:
                        for k, v in w.items():
                            if "emb_phone" in k and hasattr(v, "shape"):
                                if 768 in v.shape:
                                    is_v2 = True
                                elif 256 in v.shape:
                                    is_v2 = False
                                break

                # 3. Sampling rate & f0
                if "sr" in cpt:
                    raw_sr = cpt["sr"]
                    sr = int(str(raw_sr).replace("k", "000"))
                elif "config" in cpt and len(cpt["config"]) > 0:
                    sr = int(cpt["config"][-1])
                f0 = bool(cpt.get("f0", 1))
        except Exception as e:
            print(f"  ⚠️ Warning saat membaca metadata: {e}")

    # Konfigurasi dimensionalitas layer
    if is_v2:
        model_type = "pyTorchRVCv2"
        emb_channels = 768
        emb_output_layer = 12
        use_final_proj = False
        version_str = "v2"
    else:
        model_type = "pyTorchRVC"
        emb_channels = 256
        emb_output_layer = 9
        use_final_proj = True
        version_str = "v1"

    display_name = custom_name if custom_name else base_name.replace("-", " ").replace("_", " ").title()

    params_data = {
        "voiceChangerType": "RVC",
        "name": display_name,
        "modelFile": pth_file,
        "indexFile": index_file,
        "defaultTune": tune,
        "defaultFormantShift": 0,
        "defaultIndexRatio": 0.7,
        "defaultProtect": 0.5,
        "isONNX": False,
        "modelType": model_type,
        "f0": f0,
        "samplingRate": sr,
        "embChannels": emb_channels,
        "embOutputLayer": emb_output_layer,
        "useFinalProj": use_final_proj,
        "embedder": "hubert_base",
        "version": version_str,
        "slotIndex": slot_num,
        "speakers": {"0": "target"}
    }
    with open(os.path.join(slot_folder, "params.json"), "w", encoding="utf-8") as pf:
        json.dump(params_data, pf, indent=2)

    print(f"🎉 Slot {slot_num} SIAP! -> {display_name}")
    print(f"   • Tipe Arsitektur: {'RVC v2 (768-dim, Layer 12)' if is_v2 else 'RVC v1 (256-dim, Layer 9)'}")
    print(f"   • Weights (.pth) : {pth_file}")
    print(f"   • Feature (.index): {index_file if index_file else '(Tidak ada)'}")
    print(f"   • Sampling Rate  : {sr} Hz")
    print(f"   • Default Tune   : +{tune}")

# Eksekusi instalasi
print("=======================================================")
print("🚀 MEMULAI AUTO-INSTALL RVC MODEL KE VOICE CHANGER...")
print("=======================================================")

if INSTALL_MYTHIA and MYTHIA_URL.strip():
    install_rvc_model(MYTHIA_URL, MYTHIA_SLOT, MYTHIA_TUNE, custom_name="Mythia Batford")

if INSTALL_ZETA and ZETA_URL.strip():
    install_rvc_model(ZETA_URL, ZETA_SLOT, ZETA_TUNE, custom_name="Vestia Zeta")

if CUSTOM_URL and CUSTOM_URL.strip():
    install_rvc_model(CUSTOM_URL, CUSTOM_SLOT, CUSTOM_TUNE)

print("\n" + "="*55)
print("✨ SEMUA MODEL BERHASIL TERPASANG & TERKALIBRASI!")
print("👉 Sekarang jalankan CELL 2 untuk membuka WebUI Voice Changer!")
print("="*55)


🚀 MEMULAI AUTO-INSTALL RVC MODEL KE VOICE CHANGER...

⚡ [Slot 0] Mengunduh MythiaBatford.zip (Google Datacenter Speed)...
📦 [Slot 0] Mengekstrak file zip...
🎉 Slot 0 SIAP! -> Mythia Batford
   • Tipe Arsitektur: RVC v2 (768-dim)
   • Weights (.pth) : MythiaBatford.pth
   • Feature (.index): added_IVF840_Flat_nprobe_1_MythiaBatford_v2.index
   • Sampling Rate  : 32000 Hz
   • Default Tune   : +12

⚡ [Slot 1] Mengunduh zetaTest.zip (Google Datacenter Speed)...
📦 [Slot 1] Mengekstrak file zip...
🎉 Slot 1 SIAP! -> Vestia Zeta
   • Tipe Arsitektur: RVC v2 (768-dim)
   • Weights (.pth) : zetaTest.pth
   • Feature (.index): added_IVF462_Flat_nprobe_1_zetaTest_v2.index
   • Sampling Rate  : 40000 Hz
   • Default Tune   : +12

✨ SEMUA MODEL BERHASIL TERPASANG & TERKALIBRASI!
👉 Sekarang jalankan CELL 2 untuk membuka WebUI Voice Changer!


In [ ]:
# ==============================================================================
# 🚀 CELL 2: LAUNCH REALTIME VOICE CHANGER WEBUI (AUTO CLOUDFLARE TUNNEL)
# ==============================================================================
import os
import re
import json
import time
import socket
import threading
import subprocess
import torch
from dotenv import set_key

TUNNEL = "CLOUDFLARE (Rekomendasi - Free Tanpa Akun)" #@param ["CLOUDFLARE (Rekomendasi - Free Tanpa Akun)", "NGROK"]
NGROK_TOKEN = "" #@param {type:"string"}
NGROK_REGION = "ap" #@param ["ap", "us", "eu", "au", "in", "jp", "sa"]

PORT = 18888
os.chdir("/content/ML_Program")

# 🔧 1. FIX UBUNTU 24.04 (NOBLE) LIBSTDC++ COMPATIBILITY FOR SOUNDDEVICE / LIBPORTAUDIO
internal_dir = "/content/ML_Program/_internal"
if os.path.exists(internal_dir):
    for f in os.listdir(internal_dir):
        if "libstdc++.so" in f and not os.path.islink(os.path.join(internal_dir, f)):
            try:
                os.remove(os.path.join(internal_dir, f))
            except Exception:
                pass
    if not os.path.exists(os.path.join(internal_dir, "libstdc++.so.6")) and os.path.exists("/usr/lib/x86_64-linux-gnu/libstdc++.so.6"):
        try:
            os.symlink("/usr/lib/x86_64-linux-gnu/libstdc++.so.6", os.path.join(internal_dir, "libstdc++.so.6"))
        except Exception:
            pass

if os.path.exists("/usr/lib/x86_64-linux-gnu/libstdc++.so.6"):
    os.environ["LD_PRELOAD"] = "/usr/lib/x86_64-linux-gnu/libstdc++.so.6"

# 🧠 2. AUTO-KALIBRASI DIMENSI & LAYER DI SEMUA SLOT (v1 256-dim vs v2 768-dim)
model_base_dir = "/content/ML_Program/model_dir"
if os.path.exists(model_base_dir):
    for slot_folder in sorted(os.listdir(model_base_dir)):
        full_slot = os.path.join(model_base_dir, slot_folder)
        params_file = os.path.join(full_slot, "params.json")
        if os.path.isdir(full_slot) and os.path.exists(params_file):
            pths = [f for f in os.listdir(full_slot) if f.endswith(".pth")]
            if pths:
                pth_path = os.path.join(full_slot, pths[0])
                try:
                    cpt = torch.load(pth_path, map_location="cpu")
                    if isinstance(cpt, dict):
                        is_v2 = False
                        ver = str(cpt.get("version", "")).lower()
                        if ver == "v2":
                            is_v2 = True

                        if "weight" in cpt:
                            w = cpt["weight"]
                            for k in ["enc_p.emb_phone.weight", "enc_p.text_embedding.weight"]:
                                if k in w and hasattr(w[k], "shape"):
                                    if 768 in w[k].shape:
                                        is_v2 = True
                                    elif 256 in w[k].shape:
                                        is_v2 = False
                                    break
                            else:
                                for k, v in w.items():
                                    if "emb_phone" in k and hasattr(v, "shape"):
                                        if 768 in v.shape:
                                            is_v2 = True
                                        elif 256 in v.shape:
                                            is_v2 = False
                                        break

                        sr = cpt.get("sr", 40000)
                        if "config" in cpt and len(cpt["config"]) > 0:
                            sr = cpt["config"][-1]

                        with open(params_file, "r", encoding="utf-8") as pf:
                            pdata = json.load(pf)

                        # Kalibrasi parameter dimensionalitas & layer embedder secara utuh
                        pdata["modelType"] = "pyTorchRVCv2" if is_v2 else "pyTorchRVC"
                        pdata["embChannels"] = 768 if is_v2 else 256
                        pdata["embOutputLayer"] = 12 if is_v2 else 9
                        pdata["useFinalProj"] = False if is_v2 else True
                        pdata["version"] = "v2" if is_v2 else "v1"
                        pdata["embedder"] = "hubert_base"
                        pdata["samplingRate"] = int(sr)

                        with open(params_file, "w", encoding="utf-8") as pf:
                            json.dump(pdata, pf, indent=2)
                        print(f"🎯 Slot {slot_folder} ({pdata.get('name', slot_folder)}): Terkalibrasi {'RVC v2 (768-dim, Layer 12)' if is_v2 else 'RVC v1 (256-dim, Layer 9)'} - {sr}Hz ✅")
                except Exception as e:
                    pass

# ⚡ 3. PRE-CONFIG STORED_SETTING.JSON (FIX CRITICAL TORCHSCRIPT JIT BUG & ACTIVATE CUDA)
stored_setting_path = "/content/ML_Program/stored_setting.json"
base_settings = {}
if os.path.exists(stored_setting_path):
    try:
        with open(stored_setting_path, "r", encoding="utf-8") as f:
            base_settings = json.load(f)
    except Exception:
        pass

base_settings.update({
    "gpu": 0,                     # Otomatis gunakan NVIDIA Tesla T4 (CUDA:0)
    "disableJit": 1,              # ⚡ MATIKAN JIT TORCHSCRIPT (Memperbaiki error tensor [1, 256] vs [1, 768])
    "forceFp32": 0,               # Gunakan akselerasi FP16 GPU
    "f0Detector": "rmvpe",        # Pitch extractor paling jernih & akurat
    "serverReadChunkSize": 128,   # Latency rendah (~100ms)
    "extraConvertSize": 0.5,
    "crossFadeOverlapSize": 0.1,
    "modelSlotIndex": 0,          # Pre-select Slot 0 (Mythia Batford)
    "dstId": 0,
    "tran": 12,                   # Pitch default +12
    "indexRatio": 0.7,
    "protect": 0.5
})

with open(stored_setting_path, "w", encoding="utf-8") as f:
    json.dump(base_settings, f, indent=2)
print("🚀 Hardware Accelerated: GPU=CUDA:0 (Tesla T4) | TorchScript JIT Fix Applied | f0=rmvpe ✅")

# 4. Konfigurasi .env (Izinkan semua origin dan matikan sample mode)
set_key('.env', "SAMPLE_MODE", "")
set_key('.env', "ALLOWED_ORIGINS", '["*"]')

public_url = ""

# 5. Inisialisasi Tunnel
if "CLOUDFLARE" in TUNNEL:
    print("🌐 [1/2] Membuka jalur Cloudflare Tunnel (100% Free & No Token)...")
    p_cf = subprocess.Popen(["/content/cloudflared", "tunnel", "--protocol", "http2", "--url", f"http://127.0.0.1:{PORT}"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

    for line in p_cf.stdout:
        if "trycloudflare.com" in line:
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
            if m:
                public_url = m.group(0)
                break

elif TUNNEL == "NGROK":
    print("🌐 [1/2] Membuka jalur Ngrok Tunnel...")
    if not NGROK_TOKEN:
        raise ValueError("❌ Masukkan NGROK_TOKEN jika memilih mode tunnel NGROK!")
    from pyngrok import conf, ngrok
    conf.get_default().auth_token = NGROK_TOKEN
    conf.get_default().region = NGROK_REGION
    ngrok_conn = ngrok.connect(PORT)
    public_url = ngrok_conn.public_url

if public_url:
    set_key('.env', "ALLOWED_ORIGINS", json.dumps([public_url, "http://localhost:18888", "http://127.0.0.1:18888", "*"]))

# Background listener: Tunggu sampai server 100% listening di port 18888 sebelum memberitahu user
def notify_ready():
    while True:
        time.sleep(1)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', PORT))
        sock.close()
        if result == 0:
            time.sleep(2)
            print("\n" + "="*60)
            print("🎉 SERVER READY! BUKA LINK VOICE CHANGER WEBUI DI BAWAH INI:")
            print(f"👉 {public_url}")
            print("="*60 + "\n")
            break

threading.Thread(target=notify_ready, daemon=True).start()

# 6. Jalankan Engine Voice Changer
print("🚀 [2/2] Meluncurkan MMVCServerSIO Engine (Tesla T4 CUDA Accelerated)...")
!LD_PRELOAD=/usr/lib/x86_64-linux-gnu/libstdc++.so.6 ./ML_Program


🎯 Slot 0 (Mythia Batford): Terkalibrasi RVC v2 (768-dim) - 32000Hz ✅
🎯 Slot 1 (Vestia Zeta): Terkalibrasi RVC v2 (768-dim) - 40000Hz ✅
🌐 [1/2] Membuka jalur Cloudflare Tunnel (100% Free & No Token)...
🚀 [2/2] Meluncurkan MMVCServerSIO Engine...
2026-09-18 15:26:10,496 INFO     [main] Python: 3.10.19 (main, Oct 10 2025, 01:02:36) [GCC 11.4.0]
2026-09-18 15:26:10,497 INFO     [main] Voice changer version: b2397 NVIDIA-CUDA
2026-09-18 15:26:10,498 INFO     [main] Server settings: model_dir='model_dir' sound_dir='sound_dir' content_vec_500_onnx_on=True host='127.0.0.1' port=18888 ssl_enabled=False ssl_certfile=None ssl_keyfile=None allowed_origins=['https://parcel-improved-congress-transcripts.trycloudflare.com', 'http://localhost:18888', 'http://127.0.0.1:18888', '*'] edition='NVIDIA-CUDA'
2026-09-18 15:26:10,498 INFO     [main] Checking for mandatory models...
2026-09-18 15:26:10,697 INFO     [ModelManager] All mandatory models are already downloaded.
2026-09-18 15:26:17,077 INFO     [_l